# What does Qwen's real chat template ask the model to write?

Chapter 8 · Day 11

Notebooks 01–02 used a small symbolic template (`BOS/SYSTEM/USER/ASSISTANT/END`) so every
token could be inspected. Real training code starts somewhere else: a list of messages, a
model's own chat template, and a `labels` tensor in which every ungraded position is `-100`.
This notebook makes that step with the actual **Qwen3-0.6B tokenizer**.

It needs the Qwen3-0.6B tokenizer files in the local Hugging Face cache. They are loaded with
`local_files_only=True`, so nothing is downloaded. No model weights are loaded and nothing is
trained. The logits in Predict 5 are random numbers used only to check the loss arithmetic.

Use the cycle: prediction → inspect mechanism → change one choice → interpret limits.
Runnable reference answers follow each question. All computations use CPU.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import transformers
from transformers import AutoTokenizer

MODEL = "Qwen/Qwen3-0.6B"
try:
    tok = AutoTokenizer.from_pretrained(MODEL, local_files_only=True)
except OSError as error:
    raise SystemExit(f"{MODEL} tokenizer is not in the local Hugging Face cache; "
                     "this lesson never downloads it.") from error
print("transformers", transformers.__version__, "| tokenizer:", tok.name_or_path)

messages = [
    {"role": "system",    "content": "You are a helpful assistant."},
    {"role": "user",      "content": "Give me one word for a very happy feeling."},
    {"role": "assistant", "content": "Elated."},
    {"role": "user",      "content": "Another one?"},
    {"role": "assistant", "content": "Joyful."},
]

def render_ids(msgs, **kwargs):
    """Token IDs exactly as the model's own chat template produces them."""
    return list(tok.apply_chat_template(msgs, return_dict=True, **kwargs)["input_ids"])

def show(ids, labels=None):
    """One row per token: position, token text, and whether its label is graded."""
    for i, t in enumerate(ids):
        piece = repr(tok.decode([t]))
        mark = "" if labels is None else ("graded" if labels[i] != -100 else "-")
        print(f"{i:>3}  {piece:<22}{mark}")

### Prediction

## Predict 1

Render the five-message conversation with Qwen3's template. Will the two assistant turns, `Elated.` and `Joyful.`, look the same in the training string?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
text = tok.apply_chat_template(messages, tokenize=False)
print(text)
ids = render_ids(messages)
assert ids == tok(text, add_special_tokens=False)["input_ids"]
print(len(ids), "tokens")

### Reference explanation

### Reference answer and mechanism

No. The first answer is rendered as `<|im_start|>assistant\nElated.<|im_end|>`, but the last
one gets an empty reasoning block first:
`<|im_start|>assistant\n<think>\n\n</think>\n\nJoyful.<|im_end|>`.

Qwen3 supports a thinking mode. Its template strips reasoning from earlier assistant turns and
writes an empty `<think></think>` block into the final one. So the same conversation contains
two different answer formats. Rendering with the model's own template, never by hand, is the
only way to see what the model is really trained on. Template behavior can change between
model and library versions, so re-render whenever either changes.

### Prediction

## Predict 2

`apply_chat_template` has an option, `return_assistant_tokens_mask=True`, that promises a mask of the assistant tokens. How many tokens will it mark for this conversation?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
builtin = tok.apply_chat_template(messages, return_dict=True, return_assistant_tokens_mask=True)
print("tokens marked as assistant:", sum(builtin["assistant_masks"]), "of", len(builtin["input_ids"]))

### Reference explanation

### Reference answer and mechanism

Zero. The option only works when the template wraps assistant text in `{% generation %}`
markers, and Qwen3's template doesn't have them. The library prints a warning and returns an
all-zero mask; it doesn't raise an error.

Built from this mask, every label would be `-100`. Depending on the training code, that
produces a NaN loss (an average over zero tokens) or quietly trains nothing. The lesson is
general: a mask is a claim about the data, so check it, for example by counting and decoding
the graded tokens, before trusting it.

### Prediction

## Predict 3

Build the labels yourself. Grade each assistant answer together with its closing `<|im_end|>`, and nothing else. Before running: should the empty `<think>\n\n</think>\n\n` block be graded? And the `\n` that follows each `<|im_end|>`? Try writing `build_labels(ids)` in a scratch cell first.

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
HEADER = tok.encode("<|im_start|>assistant\n", add_special_tokens=False)
EMPTY_THINK = tok.encode("<think>\n\n</think>\n\n", add_special_tokens=False)
IM_END = tok.convert_tokens_to_ids("<|im_end|>")

def build_labels(ids, grade_think_scaffold=False):
    """Copy assistant answer tokens and their <|im_end|> into labels; -100 elsewhere."""
    labels, spans, i = [-100] * len(ids), [], 0
    while i < len(ids):
        if ids[i:i + len(HEADER)] == HEADER:
            start = i + len(HEADER)
            if not grade_think_scaffold and ids[start:start + len(EMPTY_THINK)] == EMPTY_THINK:
                start += len(EMPTY_THINK)
            end = ids.index(IM_END, start)          # the answer's own closing marker
            labels[start:end + 1] = ids[start:end + 1]
            spans.append((start, end))
            i = end + 1
        else:
            i += 1
    return labels, spans

labels, spans = build_labels(ids)
show(ids, labels)
graded = [tok.decode(ids[s:e + 1]) for s, e in spans]
print("graded spans:", graded)
assert graded == ["Elated.<|im_end|>", "Joyful.<|im_end|>"]
assert all(labels[e] == IM_END for _, e in spans)        # every answer learns to stop
assert labels[-1] == -100                                 # trailing newline is not graded
print("graded tokens:", sum(l != -100 for l in labels))

def plot_labels(ids, labels):
    """Draw the rendered conversation, one message per row; graded tokens in blue."""
    rows, row = [], []
    for t, l in zip(ids, labels):
        row.append((tok.decode([t]).replace("\n", "\\n"), l != -100))
        if t == IM_END:
            rows.append(row); row = []
    if row:                                   # trailing newline after the last <|im_end|>
        rows[-1].extend(row)
    fig, ax = plt.subplots(figsize=(12, 0.62 * len(rows) + 0.9))
    for r, cells in enumerate(rows):
        x = 0.0
        for piece, graded in cells:
            w = 0.11 * len(piece) + 0.22
            ax.add_patch(plt.Rectangle((x, -r - 0.38), w, 0.76, edgecolor="#CBD5E1", linewidth=1,
                                       facecolor="#2563EB" if graded else "#F1F5F9"))
            ax.text(x + w / 2, -r, piece, ha="center", va="center", fontsize=9,
                    family="DejaVu Sans Mono", color="white" if graded else "#111827")
            x += w + 0.05
    ax.set(xlim=(-0.1, 15.5), ylim=(-len(rows) + 0.4, 0.6), xticks=[], yticks=[])
    ax.set_title("Qwen3 chat template: graded label tokens in blue (all others are -100)",
                 loc="left", fontsize=11)
    for side in ax.spines.values():
        side.set_visible(False)
    plt.tight_layout()
    plt.show()
    return fig

plot_labels(ids, labels);

### Reference explanation

### Reference answer and mechanism

Eight tokens are graded: `El`, `ated`, `.`, `<|im_end|>` and `Joy`, `ful`, `.`, `<|im_end|>`.
Everything else is `-100`: the system and user messages, every `<|im_start|>assistant\n`
header, and the `\n` after each `<|im_end|>`. That newline is template formatting after the
turn has already ended, and the model stops generating at `<|im_end|>`, so it never has to
produce it.

The empty think block is a **design decision**, not a fixed rule. This reference leaves it
ungraded because, in thinking-off inference, the template itself places that block in the
prompt (Predict 4 checks this). The model never has to generate it. A guiding principle:
grade exactly the tokens the model will have to produce at inference. Set
`grade_think_scaffold=True` to see the alternative, which teaches the model to emit an
empty reasoning block itself.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![05 Qwen chat template and labels: reference plot 01](../figures/chapter-08/day-11-05_qwen_chat_template_and_labels-01.png)

This fixed image comes from the CPU reference with the cached Qwen3-0.6B tokenizer; changing inputs does not update the preview automatically.

### Prediction

## Predict 4

At inference with thinking off, the prompt is built with `add_generation_prompt=True, enable_thinking=False`. For each answer, does the training prefix before it match that inference prompt token for token?

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
for turn, (start, _) in enumerate(spans):
    history = messages[:2 + 2 * turn]                  # everything before this answer
    off = render_ids(history, add_generation_prompt=True, enable_thinking=False)
    on = render_ids(history, add_generation_prompt=True)
    print(f"answer {turn + 1}: matches thinking-off prompt: {ids[:start] == off}; "
          f"matches thinking-on prompt: {ids[:start] == on}")

### Reference explanation

### Reference answer and mechanism

Only the final answer's prefix matches the thinking-off inference prompt. The first answer was
trained after a bare `<|im_start|>assistant\n`, which is the thinking-*on* prompt. In
thinking-on generation, though, the model would normally open with `<think>`. So the earlier
turn teaches a context the model won't see in either mode exactly as trained.

This is why multi-turn SFT pipelines often turn one conversation into **one example per
assistant turn**. Each example renders the history up to that turn and grades only its final
answer, so every graded answer has the inference-time prefix. Another option is to grade only
the last turn. Both are dataset-design choices worth checking against the template, not
defaults to assume.

### Prediction

## Predict 5

Pass these labels to cross-entropy with `ignore_index=-100`. How many positions contribute to the loss, and does the result equal your own average of −log p(target) over the graded positions? Remember the one-position shift: the logit at position t predicts the token at t+1.

Pause and explain your reasoning before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction or attempt.

In [ ]:
torch.manual_seed(0)
logits = torch.randn(len(ids), len(tok))       # random stand-in for model outputs
y = torch.tensor(labels)

loss = F.cross_entropy(logits[:-1], y[1:], ignore_index=-100)   # shift: logit t predicts t+1
keep = y[1:] != -100
manual = -torch.log_softmax(logits[:-1], dim=-1)[keep].gather(1, y[1:][keep][:, None]).mean()
print("positions in the loss:", int(keep.sum()), "| library:", float(loss), "| manual:", float(manual))
assert torch.allclose(loss, manual)

off_by_one = F.cross_entropy(logits, y, ignore_index=-100)      # forgot the shift
print("without the shift:", float(off_by_one), "(same shapes, wrong task)")

### Reference explanation

### Reference answer and mechanism

Eight positions contribute, one per graded label, and the library value equals the manual
average exactly. `ignore_index=-100` removes those positions from the sum *and* from the
count, so the average is over graded tokens only.

The last line forgets the shift. It still runs and returns a number, because the shapes are
valid, but now each logit is trained to predict its *own* input token, which is a copying
task. That's the silent off-by-one from notebook 01's Predict 3, now in real training code.
Hugging Face causal-LM models shift `labels` internally, so pass unshifted labels to them and
check once per codebase which convention applies.

## Reading the evidence

**What this establishes:** for this cached Qwen3-0.6B tokenizer and installed `transformers`
version, the reference labels grade exactly the two answers plus their `<|im_end|>` markers.
The built-in assistant mask returns zero tokens. Only the final answer's prefix matches the
thinking-off inference prompt. `ignore_index=-100` averages over graded positions only.

**What it doesn't establish:** anything about model quality. No weights are loaded, the
logits are random, and nothing is trained. Template text is versioned behavior: other Qwen
releases, other model families or other library versions can render differently. Whether to
grade the empty think block, or every turn versus the last, is a declared design choice
here, not a measured best practice.

Back to [Chapter 8](../../book/chapters/08-instruction-data-as-an-interface.md) §8.2–8.4 for
the symbolic version of the same ideas, or forward to Day 12's
[SFT objective notebook](../day-12/01_sft_objective_and_gradient_paths.ipynb) to see that
masked prompt tokens still receive gradient through the graded answer.